In [1]:
# Import required libraries  
import os  
import json  
import openai  
import pandas as pd    
from dotenv import load_dotenv  
from tenacity import retry, wait_random_exponential, stop_after_attempt  
from azure.core.credentials import AzureKeyCredential  
from azure.search.documents import SearchClient  
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.models import Vector  
from azure.search.documents.indexes.models import (  
    SearchIndex,  
    SearchField,  
    SearchFieldDataType,  
    SimpleField,  
    SearchableField,  
    SearchIndex,  
    SemanticConfiguration,  
    PrioritizedFields,  
    SemanticField,  
    SearchField,  
    SemanticSettings,  
    VectorSearch,  
    VectorSearchAlgorithmConfiguration
)  

In [2]:
# Configure environment variables  
load_dotenv("../.env", override=True)  
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")  
index_name = "app-crg-rag-tool"  
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")  
openai.api_type = 'azure'  
openai.api_key = os.getenv("OPENAI_API_KEY")  
openai.api_base = os.getenv("OPENAI_API_BASE")  
openai.api_version = "2023-05-15"
credential = AzureKeyCredential(key)

In [3]:
# Generate Document Embeddings using OpenAI Ada 002
# input_data = pd.read_json(path_or_buf='data/labeled_constraint.jsonl', lines=True)

# Read the text-sample.json
with open('../data/rag_tools.json', 'r', encoding='utf-8') as file:
    input_data = json.load(file)

@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def generate_embeddings(text):
    response = openai.Embedding.create(
        input=text, engine="text-embedding-ada-002")
    embeddings = response['data'][0]['embedding']
    return embeddings


# Generate embeddings for title and content fields
for item in input_data:
    print(item)
    label = item['description']
    constraint = item['tool']
    label_embeddings = generate_embeddings(label)
    constraint_embeddings = generate_embeddings(constraint)
    item['descriptionVector'] = label_embeddings
    item['toolVector'] = constraint_embeddings
    item['@search.action'] = 'upload'

# Output embeddings to docVectors.json file
with open("output/toolVectors.json", "w") as f:
    json.dump(input_data, f)

{'id': '1', 'description': 'calculate maximal cliques.', 'tool': "def process_graph(graph_data):\n    return_object = {\n        'type': 'text',\n        'data': len(list(nx.find_cliques(graph_data)))\n    }\n    return return_object"}
{'id': '2', 'description': 'when you check node degrees (include max degree and min degree).', 'tool': 'degrees = [d for n, d in graph_data.degree()]'}


In [4]:
# Create a search index
index_client = SearchIndexClient(
    endpoint=service_endpoint, credential=credential)
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="description", type=SearchFieldDataType.String,
                    searchable=True, retrievable=True),
    SearchableField(name="tool", type=SearchFieldDataType.String,
                    searchable=False, retrievable=True),
    SearchField(name="descriptionVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=1536, vector_search_configuration="my-vector-config"),
    SearchField(name="toolVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=1536, vector_search_configuration="my-vector-config"),
]

vector_search = VectorSearch(
    algorithm_configurations=[
        VectorSearchAlgorithmConfiguration(
            name="my-vector-config",
            kind="hnsw",
            hnsw_parameters={
                "m": 4,
                "efConstruction": 400,
                "efSearch": 1000,
                "metric": "cosine"
            }
        )
    ]
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=PrioritizedFields(
        prioritized_keywords_fields=[SemanticField(field_name="description")],
        prioritized_content_fields=[SemanticField(field_name="tool")]
    )
)

# Create the semantic settings with the configuration
semantic_settings = SemanticSettings(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=index_name, fields=fields,
                    vector_search=vector_search, semantic_settings=semantic_settings)
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 app-crg-rag-tool created


In [5]:
# Upload some documents to the index
with open('output/toolVectors.json', 'r') as file:  
    documents = json.load(file)  
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents)  
print(f"Uploaded {len(documents)} documents")

Uploaded 2 documents


In [12]:
# Pure Vector Search
query = "List all ports contained in packet switch ju1.a1.m1.s2c1. Return a list."
search_client = SearchClient(service_endpoint, index_name, AzureKeyCredential(key))  
  
results = search_client.search(  
    search_text='',  
    vector=generate_embeddings(query),
    top_k=1,
    vector_fields="descriptionVector",
    select=["description", "tool"] 
)  

for result in results:
    print(f"Score: {result['@search.score']}")
    print(f"Label: {result['description']}")  
    print(f"Constraint: {result['tool']}\n")  


Score: 0.7674955
Label: bandwidth on each SUPERBLOCK
Constraint: 
def ground_truth_process_graph(graph_data):
    
    super_blocks = [node for node in graph_data.nodes if 'EK_SUPERBLOCK' in graph_data.nodes[node]['type']]
    bandwidths = []

    for super_block in super_blocks:
        packet_switches = [node for node in graph_data.successors(super_block) if 'EK_PACKET_SWITCH' in graph_data.nodes[node]['type']]
        total_bandwidth = 0

        for packet_switch in packet_switches:
            ports = [node for node in graph_data.successors(packet_switch) if 'EK_PORT' in graph_data.nodes[node]['type']]

            for port in ports:
                total_bandwidth += graph_data.nodes[port]['physical_capacity_bps'] / 1e6  # Convert to Mbps

        bandwidths.append([super_block, total_bandwidth])

    return_object = {
        'type': 'table',
        'data': [['SUPERBLOCK', 'Bandwidth']] + bandwidths
    }

    return return_object

